# Final Test RMSE Summary from JSON Files

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/dissertation")
else:
    PROJECT_ROOT = Path("../../..").resolve()

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"

HORIZONS = [15, 30, 60, 90, 120]
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

print("PROJECT_ROOT:", PROJECT_ROOT)

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/dissertation


In [2]:
RESULT_FILES = {
    "Model 1": (
        PROJECT_ROOT
        / "experiments"
        / "7_federated_learning"
        / "results"
        / "fed_cgm_only"
        / "run_results.json"
    ),

    "Model 2": (
        PROJECT_ROOT
        / "experiments"
        / "7_federated_learning"
        / "results"
        / "fed_cgm_time"
        / "run_results.json"
    ),

    "Model 3": (
        PROJECT_ROOT
        / "experiments"
        / "7_federated_learning"
        / "results"
        / "fed_cgm_time_image"
        / "resnet18"
        / "run_results.json"
    ),

    "Model 4": (
        PROJECT_ROOT
        / "experiments"
        / "7_federated_learning"
        / "results"
        / "fed_cgm_time_image"
        / "vit"
        / "run_results.json"
    ),
    "Model 5": (
        PROJECT_ROOT
        / "experiments"
        / "7_federated_learning"
        / "results"
        / "fed_cgm_time_image"
        / "clip"
        / "run_results.json"
    ),
    "Model 6": (
        PROJECT_ROOT
        / "experiments"
        / "7_federated_learning"
        / "results"
        / "fed_cgm_time_image"
        / "dinov2"
        / "run_results.json"
    ),
}

for name, path in RESULT_FILES.items():
    assert path.exists(), f"{name} not found: {path}"
    print(f"{name:<20} {path}")

Model 1              /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_only/run_results.json
Model 2              /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_time/run_results.json
Model 3              /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_time_image/resnet18/run_results.json
Model 4              /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_time_image/vit/run_results.json
Model 5              /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_time_image/clip/run_results.json
Model 6              /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_time_image/dinov2/run_results.json


In [3]:
def h_entry(result_dict, horizon):
    if str(horizon) in result_dict:
        return result_dict[str(horizon)]
    if horizon in result_dict:
        return result_dict[horizon]
    raise KeyError(f'Horizon {horizon} not found. Keys={list(result_dict.keys())}')


def p_entry(participant_dict, participant_id):
    if participant_id in participant_dict:
        return participant_dict[participant_id]
    pid = str(participant_id)
    if pid in participant_dict:
        return participant_dict[pid]
    raise KeyError(
        f'Participant {participant_id} not found. '
        f'Available IDs={list(participant_dict.keys())}'
    )

In [4]:
all_results = {}

for name, path in RESULT_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f'{name}: file not found: {path}')

    with open(path, 'r') as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise TypeError(
            f'{name}: expected JSON root to be a list, '
            f'found {type(data).__name__}'
        )

    all_results[name] = data
    print(f'{name:<20} {len(data)} runs loaded')

Model 1              10 runs loaded
Model 2              10 runs loaded
Model 3              10 runs loaded
Model 4              10 runs loaded
Model 5              10 runs loaded
Model 6              10 runs loaded


In [5]:
participant_ids_by_model = {}

for name, rows in all_results.items():
    seeds = [int(r['seed']) for r in rows]

    if sorted(seeds) != sorted(SEEDS):
        raise ValueError(
            f'{name}: seed mismatch. Observed={sorted(seeds)}, '
            f'Expected={sorted(SEEDS)}'
        )

    for r in rows:
        if 'test_results' not in r:
            raise KeyError(f'{name}, seed {r["seed"]}: missing test_results')
        if 'test_results_per_participant' not in r:
            raise KeyError(
                f'{name}, seed {r["seed"]}: '
                'missing test_results_per_participant'
            )

    reference_ids = sorted(rows[0]['test_results_per_participant'].keys())

    for r in rows[1:]:
        observed = sorted(r['test_results_per_participant'].keys())
        if observed != reference_ids:
            raise ValueError(
                f'{name}, seed {r["seed"]}: participant IDs differ. '
                f'Observed={observed}, Expected={reference_ids}'
            )

    participant_ids_by_model[name] = reference_ids
    print(
        f'{name:<20} {len(seeds)} seeds | '
        f'{len(reference_ids)} test participants | IDs={reference_ids}'
    )

print('\nAll within-file checks passed.')

Model 1              10 seeds | 6 test participants | IDs=['CGMacros-008', 'CGMacros-014', 'CGMacros-019', 'CGMacros-020', 'CGMacros-034', 'CGMacros-039']
Model 2              10 seeds | 6 test participants | IDs=['CGMacros-008', 'CGMacros-014', 'CGMacros-019', 'CGMacros-020', 'CGMacros-034', 'CGMacros-039']
Model 3              10 seeds | 6 test participants | IDs=['CGMacros-008', 'CGMacros-014', 'CGMacros-019', 'CGMacros-020', 'CGMacros-034', 'CGMacros-039']
Model 4              10 seeds | 6 test participants | IDs=['CGMacros-008', 'CGMacros-014', 'CGMacros-019', 'CGMacros-020', 'CGMacros-034', 'CGMacros-039']
Model 5              10 seeds | 6 test participants | IDs=['CGMacros-008', 'CGMacros-014', 'CGMacros-019', 'CGMacros-020', 'CGMacros-034', 'CGMacros-039']
Model 6              10 seeds | 6 test participants | IDs=['CGMacros-008', 'CGMacros-014', 'CGMacros-019', 'CGMacros-020', 'CGMacros-034', 'CGMacros-039']

All within-file checks passed.


In [6]:
reference_name = next(iter(participant_ids_by_model))
reference_ids = participant_ids_by_model[reference_name]

for name, ids in participant_ids_by_model.items():
    if ids != reference_ids:
        raise ValueError(
            f'Test participant mismatch: {reference_name} vs {name}. '
            f'{reference_name}={reference_ids}; {name}={ids}'
        )

TEST_PARTICIPANT_IDS = reference_ids

print('Cross-file participant check passed.')
print(f'Test participants ({len(TEST_PARTICIPANT_IDS)}): {TEST_PARTICIPANT_IDS}')

Cross-file participant check passed.
Test participants (6): ['CGMacros-008', 'CGMacros-014', 'CGMacros-019', 'CGMacros-020', 'CGMacros-034', 'CGMacros-039']


In [7]:
summary_rows = []
participant_rows = []
seed_rows = []

for name, rows in all_results.items():
    seed_dict = {int(r['seed']): r for r in rows}
    row = {'model': name}

    for h in HORIZONS:
        seed_values = np.asarray([
            float(h_entry(seed_dict[seed]['test_results'], h)['rmse'])
            for seed in SEEDS
        ])

        mean_rmse = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        for seed, value in zip(SEEDS, seed_values):
            seed_rows.append({
                'model': name,
                'seed': seed,
                'horizon': h,
                'test_rmse': float(value),
            })

        participant_means = []

        for pid in TEST_PARTICIPANT_IDS:
            pid_seed_values = np.asarray([
                float(
                    h_entry(
                        p_entry(
                            seed_dict[seed]['test_results_per_participant'],
                            pid
                        ),
                        h
                    )['rmse']
                )
                for seed in SEEDS
            ])

            pid_mean = float(np.mean(pid_seed_values))
            participant_means.append(pid_mean)

            participant_rows.append({
                'model': name,
                'participant_id': pid,
                'horizon': h,
                'mean_rmse_across_seeds': pid_mean,
                'sd_rmse_across_seeds': float(
                    np.std(pid_seed_values, ddof=1)
                ),
            })

        sd2 = float(np.std(participant_means, ddof=1))

        row[f'rmse_{h}'] = mean_rmse
        row[f'sd1_{h}'] = sd1
        row[f'sd2_{h}'] = sd2
        row[f'report_{h}'] = f'{mean_rmse:.2f} ± {sd1:.2f} ({sd2:.2f})'

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
participant_df = pd.DataFrame(participant_rows)
seed_df = pd.DataFrame(seed_rows)

print('Calculation complete.')

Calculation complete.


## 7. Dissertation-format test RMSE table

In [8]:
report_table = summary_df[
    ['model'] + [f'report_{h}' for h in HORIZONS]
].copy()

report_table.columns = [
    'Model', '15 min', '30 min', '60 min', '90 min', '120 min'
]

display(report_table)

print('\nFinal test RMSE: mean ± sd1 (sd2) mg/dL')

for _, row in summary_df.iterrows():
    print(f'\n{row["model"]}')
    for h in HORIZONS:
        print(
            f'  {h:>3} min: '
            f'{row[f"rmse_{h}"]:.3f} ± '
            f'{row[f"sd1_{h}"]:.2f} '
            f'({row[f"sd2_{h}"]:.2f}) mg/dL'
        )

,Model,15 min,30 min,60 min,90 min,120 min
0,Model 1,12.80 ± 0.58 (3.91),21.12 ± 0.58 (7.18),31.95 ± 0.45 (14.56),38.86 ± 0.42 (19.80),43.20 ± 0.47 (23.38)
1,Model 2,12.38 ± 0.31 (3.75),20.22 ± 0.31 (6.78),31.08 ± 0.23 (13.79),38.09 ± 0.31 (19.05),42.38 ± 0.35 (22.61)
2,Model 3,12.34 ± 0.28 (3.65),20.16 ± 0.27 (6.60),31.02 ± 0.23 (13.58),38.07 ± 0.28 (18.92),42.37 ± 0.34 (22.59)
3,Model 4,12.36 ± 0.29 (3.68),20.17 ± 0.30 (6.62),31.02 ± 0.26 (13.58),38.05 ± 0.31 (18.94),42.35 ± 0.31 (22.57)
4,Model 5,12.32 ± 0.28 (3.60),20.10 ± 0.27 (6.49),30.89 ± 0.21 (13.33),37.94 ± 0.27 (18.69),42.26 ± 0.30 (22.42)
5,Model 6,12.35 ± 0.30 (3.71),20.11 ± 0.31 (6.62),30.86 ± 0.29 (13.50),37.89 ± 0.36 (18.76),42.22 ± 0.40 (22.38)



Final test RMSE: mean ± sd1 (sd2) mg/dL

Model 1
   15 min: 12.802 ± 0.58 (3.91) mg/dL
   30 min: 21.116 ± 0.58 (7.18) mg/dL
   60 min: 31.949 ± 0.45 (14.56) mg/dL
   90 min: 38.857 ± 0.42 (19.80) mg/dL
  120 min: 43.204 ± 0.47 (23.38) mg/dL

Model 2
   15 min: 12.382 ± 0.31 (3.75) mg/dL
   30 min: 20.219 ± 0.31 (6.78) mg/dL
   60 min: 31.079 ± 0.23 (13.79) mg/dL
   90 min: 38.087 ± 0.31 (19.05) mg/dL
  120 min: 42.377 ± 0.35 (22.61) mg/dL

Model 3
   15 min: 12.343 ± 0.28 (3.65) mg/dL
   30 min: 20.156 ± 0.27 (6.60) mg/dL
   60 min: 31.022 ± 0.23 (13.58) mg/dL
   90 min: 38.073 ± 0.28 (18.92) mg/dL
  120 min: 42.375 ± 0.34 (22.59) mg/dL

Model 4
   15 min: 12.360 ± 0.29 (3.68) mg/dL
   30 min: 20.168 ± 0.30 (6.62) mg/dL
   60 min: 31.019 ± 0.26 (13.58) mg/dL
   90 min: 38.053 ± 0.31 (18.94) mg/dL
  120 min: 42.348 ± 0.31 (22.57) mg/dL

Model 5
   15 min: 12.318 ± 0.28 (3.60) mg/dL
   30 min: 20.098 ± 0.27 (6.49) mg/dL
   60 min: 30.890 ± 0.21 (13.33) mg/dL
   90 min: 37.937 ± 0.27 (1

## 8. Numerical summary

In [9]:
numeric_columns = ['model']
for h in HORIZONS:
    numeric_columns += [f'rmse_{h}', f'sd1_{h}', f'sd2_{h}']

display(summary_df[numeric_columns].round(4))

,model,rmse_15,sd1_15,sd2_15,rmse_30,sd1_30,sd2_30,rmse_60,sd1_60,sd2_60,rmse_90,sd1_90,sd2_90,rmse_120,sd1_120,sd2_120
0,Model 1,12.8020,0.5787,3.9063,21.1163,0.5762,7.1788,31.9489,0.4544,14.5592,38.8570,0.4222,19.8041,43.2035,0.4675,23.3793
1,Model 2,12.3824,0.3124,3.7485,20.2187,0.3114,6.7816,31.0788,0.2344,13.7870,38.0871,0.3076,19.0475,42.3770,0.3540,22.6076
2,Model 3,12.3432,0.2847,3.6456,20.1561,0.2720,6.5982,31.0222,0.2277,13.5771,38.0726,0.2805,18.9232,42.3748,0.3415,22.5884
3,Model 4,12.3598,0.2920,3.6839,20.1678,0.2971,6.6155,31.0187,0.2592,13.5798,38.0531,0.3061,18.9364,42.3483,0.3149,22.5669
4,Model 5,12.3177,0.2792,3.5965,20.0980,0.2708,6.4915,30.8897,0.2093,13.3333,37.9374,0.2699,18.6925,42.2589,0.3036,22.4206
5,Model 6,12.3486,0.3039,3.7108,20.1060,0.3052,6.6227,30.8645,0.2882,13.5012,37.8924,0.3619,18.7557,42.2191,0.4012,22.3757


## 9. Participant-level means

These participant-specific RMSE means across 10 seeds are the values used to calculate SD2.

In [10]:
participant_pivot = participant_df.pivot_table(
    index=['model', 'participant_id'],
    columns='horizon',
    values='mean_rmse_across_seeds'
)

participant_pivot.columns = [
    f'rmse_{h}' for h in participant_pivot.columns
]

display(participant_pivot.round(4))

rmse_15  rmse_30  rmse_60  rmse_90  rmse_120
model   participant_id                                              
Model 1 CGMacros-008    11.8657  18.9985  26.3884  29.9921   31.2524
        CGMacros-014     9.8699  16.8571  24.1139  26.9423   28.0627
        CGMacros-019     9.1767  14.7743  17.4119  18.4489   18.7911
        CGMacros-020    12.2903  20.7991  30.7901  36.1805   39.5962
        CGMacros-034    10.6134  15.0036  18.5181  21.8499   24.3393
        CGMacros-039    19.8896  33.9249  56.9608  72.7518   83.0768
Model 2 CGMacros-008    11.3111  17.7641  25.7534  29.9059   31.2500
        CGMacros-014     9.4680  15.9320  23.6501  26.8908   27.8435
        CGMacros-019     8.9419  14.2625  17.4064  18.3491   18.5281
        CGMacros-020    11.7196  19.3911  28.9531  34.7976   38.4396
        CGMacros-034    10.6857  15.4920  19.4027  22.2638   24.6907
        CGMacros-039    19.2307  32.5810  55.2090  70.9261   81.0886
Model 3 CGMacros-008    11.3068  17.7301  25.7333  30.0428   31.5516
        CGMacros-014     9.4869  15.9853  23.5729  26.5065   27.2538
        CGMacros-019     9.0154  14.4451  17.8881  18.8224   18.7687
        CGMacros-020    11.6864  19.1872  28.5012  34.3737   38.0487
        CGMacros-034    10.7678  15.7394  19.8144  22.7433   25.0985
        CGMacros-039    19.0156  32.2616  54.9904  70.8981   81.1743
Model 4 CGMacros-008    11.3282  17.8119  25.8544  29.9932   31.3112
        CGMacros-014     9.4718  15.9429  23.6835  26.8827   27.7833
        CGMacros-019     8.9647  14.3248  17.5170  18.3910   18.4942
        CGMacros-020    11.6967  19.2461  28.7039  34.6309   38.3143
        CGMacros-034    10.7664  15.7874  19.8300  22.5631   24.8648
        CGMacros-039    19.0897  32.2721  54.8831  70.7624   81.0177
Model 5 CGMacros-008    11.3492  17.8606  25.9873  30.3184   31.7049
        CGMacros-014     9.4571  15.8933  23.5427  26.6681   27.5150
        CGMacros-019     9.0031  14.4191  17.9194  18.8494   18.7737
        CGMacros-020    11.5687  18.9066  27.9886  33.8796   37.7024
        CGMacros-034    10.9206  16.0388  20.2453  22.9602   25.0926
        CGMacros-039    18.8964  32.0466  54.5335  70.4477   80.8288
Model 6 CGMacros-008    11.3058  17.7547  25.7548  30.0258   31.4211
        CGMacros-014     9.4480  15.8954  23.6434  26.9426   27.8816
        CGMacros-019     8.9263  14.2660  17.5021  18.4862   18.6580
        CGMacros-020    11.6625  19.1047  28.3452  34.2036   37.9927
        CGMacros-034    10.7357  15.7347  19.7743  22.6170   24.8845
        CGMacros-039    19.1274  32.2360  54.6432  70.3771   80.6455

In [11]:
# ============================================================
# Seed-wise improvement analysis: Model 3/4 vs Model 2
# ============================================================

REFERENCE_MODEL = "Model 2"
COMPARE_MODELS = ["Model 3", "Model 4", "Model 5", "Model 6"]

# Convert each model's list of runs into {seed: result}
seed_results = {
    model: {int(r["seed"]): r for r in rows}
    for model, rows in all_results.items()
}

improvement_rows = []

for model in COMPARE_MODELS:
    for h in HORIZONS:
        for seed in SEEDS:

            ref_rmse = float(
                h_entry(
                    seed_results[REFERENCE_MODEL][seed]["test_results"],
                    h
                )["rmse"]
            )

            model_rmse = float(
                h_entry(
                    seed_results[model][seed]["test_results"],
                    h
                )["rmse"]
            )

            delta = model_rmse - ref_rmse

            improvement_rows.append({
                "model": model,
                "horizon": h,
                "seed": seed,
                "model2_rmse": ref_rmse,
                "model_rmse": model_rmse,
                "delta_rmse": delta,
                "improved": delta < 0,
            })

improvement_df = pd.DataFrame(improvement_rows)

display(improvement_df)

,model,horizon,seed,model2_rmse,model_rmse,delta_rmse,improved
0,Model 3,15,7,12.351110,12.319719,-0.031391,True
1,Model 3,15,14,12.520571,12.504935,-0.015636,True
2,Model 3,15,21,12.495320,12.393419,-0.101901,True
3,Model 3,15,28,12.068973,12.118368,0.049396,False
4,Model 3,15,35,12.333478,12.290921,-0.042557,True
...,...,...,...,...,...,...,...
195,Model 6,120,42,42.051441,41.807771,-0.243670,True
196,Model 6,120,49,42.077744,41.981313,-0.096431,True
197,Model 6,120,56,42.726104,42.558340,-0.167764,True
198,Model 6,120,63,42.171850,41.926570,-0.245280,True


In [12]:
improvement_summary = (
    improvement_df
    .groupby(["model", "horizon"])
    .agg(
        improved_seeds=("improved", "sum"),
        total_seeds=("seed", "count"),
        mean_delta_rmse=("delta_rmse", "mean"),
        median_delta_rmse=("delta_rmse", "median"),
    )
    .reset_index()
)

improvement_summary["improvement_rate"] = (
    improvement_summary["improved_seeds"]
    / improvement_summary["total_seeds"]
    * 100
)

display(improvement_summary)

,model,horizon,improved_seeds,total_seeds,mean_delta_rmse,median_delta_rmse,improvement_rate
0,Model 3,15,9,10,-0.039206,-0.035547,90.0
1,Model 3,30,8,10,-0.062569,-0.078170,80.0
2,Model 3,60,7,10,-0.056556,-0.065460,70.0
3,Model 3,90,7,10,-0.014538,-0.039412,70.0
4,Model 3,120,6,10,-0.002210,-0.045501,60.0
5,Model 4,15,8,10,-0.022658,-0.024917,80.0
6,Model 4,30,8,10,-0.050896,-0.059414,80.0
7,Model 4,60,7,10,-0.060036,-0.082692,70.0
8,Model 4,90,8,10,-0.034029,-0.035925,80.0
9,Model 4,120,6,10,-0.028706,-0.028790,60.0


In [13]:
for model in COMPARE_MODELS:

    print(f"\n{'=' * 60}")
    print(f"{model} vs {REFERENCE_MODEL}")
    print("ΔRMSE < 0 means improvement")
    print("=" * 60)

    model_df = improvement_df[
        improvement_df["model"] == model
    ]

    delta_table = model_df.pivot(
        index="seed",
        columns="horizon",
        values="delta_rmse"
    )

    display(delta_table.round(3))

    count_table = (
        model_df
        .groupby("horizon")["improved"]
        .agg(["sum", "count"])
    )

    count_table["rate_%"] = (
        count_table["sum"] / count_table["count"] * 100
    )

    count_table.columns = [
        "Improved seeds",
        "Total seeds",
        "Improvement rate (%)"
    ]

    display(count_table.round(1))


Model 3 vs Model 2
ΔRMSE < 0 means improvement


horizon,15,30,60,90,120
seed,,,,,
7,-0.031,-0.068,-0.128,-0.107,-0.065
14,-0.016,-0.089,-0.146,-0.094,-0.064
21,-0.102,-0.066,-0.021,-0.006,0.006
28,0.049,0.111,0.231,0.277,0.233
35,-0.043,-0.123,-0.148,-0.090,-0.070
42,-0.014,-0.016,0.015,0.088,0.102
49,-0.110,-0.144,-0.001,0.154,0.227
56,-0.040,-0.108,-0.110,-0.073,-0.027
63,-0.016,0.040,0.032,-0.003,-0.087


,Improved seeds,Total seeds,Improvement rate (%)
horizon,,,
15,9,10,90.0
30,8,10,80.0
60,7,10,70.0
90,7,10,70.0
120,6,10,60.0



Model 4 vs Model 2
ΔRMSE < 0 means improvement


horizon,15,30,60,90,120
seed,,,,,
7,-0.037,-0.060,-0.088,-0.065,-0.077
14,-0.042,-0.085,-0.087,-0.023,0.026
21,-0.014,0.019,0.000,-0.010,-0.005
28,0.035,0.041,0.030,0.017,0.001
35,-0.026,-0.068,-0.078,-0.049,-0.053
42,-0.008,-0.055,-0.056,-0.016,0.003
49,-0.066,-0.083,0.055,0.164,0.181
56,-0.023,-0.053,-0.089,-0.149,-0.189
63,0.010,-0.059,-0.141,-0.114,-0.086


...

horizon,15,30,60,90,120
seed,,,,,
7,-0.037,-0.098,-0.216,-0.232,-0.248
14,-0.133,-0.213,-0.299,-0.258,-0.259
21,-0.112,-0.113,-0.152,-0.131,-0.111
28,-0.005,0.014,-0.026,-0.022,-0.001
35,-0.076,-0.200,-0.298,-0.254,-0.251
42,-0.039,-0.065,-0.066,0.054,0.137
49,-0.117,-0.198,-0.209,-0.125,-0.044
56,-0.001,-0.057,-0.133,-0.143,-0.117
63,-0.038,-0.078,-0.158,-0.115,-0.087


,Improved seeds,Total seeds,Improvement rate (%)
horizon,,,
15,10,10,100.0
30,9,10,90.0
60,10,10,100.0
90,9,10,90.0
120,9,10,90.0



Model 6 vs Model 2
ΔRMSE < 0 means improvement


horizon,15,30,60,90,120
seed,,,,,
7,0.016,-0.002,-0.054,-0.055,-0.036
14,-0.070,-0.205,-0.325,-0.229,-0.153
21,-0.054,-0.042,-0.066,-0.056,-0.038
28,-0.002,-0.045,-0.114,-0.096,-0.075
35,-0.024,-0.140,-0.257,-0.239,-0.207
42,-0.069,-0.200,-0.335,-0.301,-0.244
49,-0.037,-0.119,-0.165,-0.138,-0.096
56,-0.046,-0.153,-0.271,-0.232,-0.168
63,-0.024,-0.100,-0.255,-0.267,-0.245


,Improved seeds,Total seeds,Improvement rate (%)
horizon,,,
15,9,10,90.0
30,10,10,100.0
60,10,10,100.0
90,10,10,100.0
120,10,10,100.0


## 10. Save summaries

In [14]:
OUTPUT_DIR = Path('./test_rmse_summary')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUTPUT_DIR / 'four_json_test_rmse_summary.csv'
report_path = OUTPUT_DIR / 'four_json_test_rmse_report_table.csv'
participant_path = OUTPUT_DIR / 'four_json_test_participant_rmse.csv'
seed_path = OUTPUT_DIR / 'four_json_test_seed_rmse.csv'

summary_df.to_csv(summary_path, index=False)
report_table.to_csv(report_path, index=False)
participant_df.to_csv(participant_path, index=False)
seed_df.to_csv(seed_path, index=False)

print('Saved:')
print(' ', summary_path)
print(' ', report_path)
print(' ', participant_path)
print(' ', seed_path)

Saved:
  test_rmse_summary/four_json_test_rmse_summary.csv
  test_rmse_summary/four_json_test_rmse_report_table.csv
  test_rmse_summary/four_json_test_participant_rmse.csv
  test_rmse_summary/four_json_test_seed_rmse.csv
